### Ultrasonic sensor test example ###

In [1]:
%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import time
import os
import sys
import math
import random
import numpy as np
import cv2
from PIL import Image
from IPython.display import display, clear_output
import ipywidgets as widgets
from pynq import Overlay, MMIO, PL, allocate
from pynq.lib.video import *
from pynq_dpu import DpuOverlay
import spidev
import colorsys


# Load the overlay
overlay = DpuOverlay("../dpu/dpu.bit")

# # Load YOLOv3
overlay.load_model("../xmodel/tiny-yolov3_256.xmodel")

In [5]:
for k, v in overlay.ip_dict.items():
    if 'phys_addr' in v and v['phys_addr'] == 0xB0000000:
        print(k, v)

SONIC_2_IP_0 {'type': 'user.org:user:SONIC_2_IP:1.0', 'mem_id': 'S00_AXI', 'memtype': 'REGISTER', 'gpio': {}, 'interrupts': {}, 'parameters': {'C_S00_AXI_DATA_WIDTH': '32', 'C_S00_AXI_ADDR_WIDTH': '5', 'Component_Name': 'design_1_SONIC_2_IP_0_2', 'EDK_IPTYPE': 'PERIPHERAL', 'C_S00_AXI_BASEADDR': '0xB0000000', 'C_S00_AXI_HIGHADDR': '0xB000FFFF', 'WIZ_DATA_WIDTH': '32', 'WIZ_NUM_REG': '8', 'SUPPORTS_NARROW_BURST': '0', 'DATA_WIDTH': '32', 'PROTOCOL': 'AXI4LITE', 'FREQ_HZ': '50000000', 'ID_WIDTH': '0', 'ADDR_WIDTH': '5', 'AWUSER_WIDTH': '0', 'ARUSER_WIDTH': '0', 'WUSER_WIDTH': '0', 'RUSER_WIDTH': '0', 'BUSER_WIDTH': '0', 'READ_WRITE_MODE': 'READ_WRITE', 'HAS_BURST': '0', 'HAS_LOCK': '0', 'HAS_PROT': '1', 'HAS_CACHE': '0', 'HAS_QOS': '0', 'HAS_REGION': '0', 'HAS_WSTRB': '1', 'HAS_BRESP': '1', 'HAS_RRESP': '1', 'NUM_READ_OUTSTANDING': '8', 'NUM_WRITE_OUTSTANDING': '8', 'MAX_BURST_LENGTH': '1', 'PHASE': '0.0', 'CLK_DOMAIN': 'design_1_clk_wiz_0_0_clk_out1', 'NUM_READ_THREADS': '4', 'NUM_WRITE

In [ ]:
import time
from IPython.display import clear_output

# 현재 overlay에 올라간 IP 이름에 맞춰 사용
sonic = overlay.SONIC_2_IP_0.mmio
sonic_base_addr = sonic.base_addr

print(f"SONIC base address: 0x{sonic_base_addr:08X}")

# 50 MHz clock 기준
CLK_FREQ = 50_000_000
SOUND_SPEED_CM_S = 34300.0

def ticks_to_cm(count):
    return count * SOUND_SPEED_CM_S / (2.0 * CLK_FREQ)

try:
    while True:
        # 새 구조: 제어 레지스터 없이 5개 센서 값만 읽음
        val1 = sonic.read(0x00)
        val2 = sonic.read(0x04)
        val3 = sonic.read(0x08)
        val4 = sonic.read(0x0C)
        val5 = sonic.read(0x10)

        dist1 = ticks_to_cm(val1)
        dist2 = ticks_to_cm(val2)
        dist3 = ticks_to_cm(val3)
        dist4 = ticks_to_cm(val4)
        dist5 = ticks_to_cm(val5)

        print(f"val1 : {val1:10d}  ->  {dist1:7.2f} cm")
        print(f"val2 : {val2:10d}  ->  {dist2:7.2f} cm")
        print(f"val3 : {val3:10d}  ->  {dist3:7.2f} cm")
        print(f"val4 : {val4:10d}  ->  {dist4:7.2f} cm")
        print(f"val5 : {val5:10d}  ->  {dist5:7.2f} cm")

        time.sleep(0.1)
        clear_output(wait=True)

except KeyboardInterrupt:
    print("Interrupt detected, shutting down gracefully...")

print("done")

val1 :      12499  ->     4.29 cm
val2 :    2359023  ->   809.14 cm
val3 :          1  ->     0.00 cm
val4 :     177860  ->    61.01 cm
val5 :          1  ->     0.00 cm
